# BRCA all omics benchmark
- benchmark for the brca dataset
- take
    - [x] mRNA
    - [x] CNA
    - [x] miRNA
    - [ ] DNA methylation data
- benchmark several prediction methods on them, using
    - [ ] early integration
    - [ ] late integration

In [8]:
%load_ext autoreload
%autoreload 2

In [9]:
import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variational_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval

In [3]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

In [4]:
labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
labels, y

(shape: (483, 2)
 ┌─────────────────┬───────────────────────┐
 │ sampleID        ┆ PAM50_mRNA_nature2012 │
 │ ---             ┆ ---                   │
 │ str             ┆ str                   │
 ╞═════════════════╪═══════════════════════╡
 │ TCGA-A1-A0SD-01 ┆ Luminal A             │
 │ TCGA-A1-A0SE-01 ┆ Luminal A             │
 │ TCGA-A1-A0SH-01 ┆ Luminal A             │
 │ TCGA-A1-A0SJ-01 ┆ Luminal A             │
 │ TCGA-A1-A0SK-01 ┆ Basal-like            │
 │ …               ┆ …                     │
 │ TCGA-E2-A1B4-01 ┆ Luminal A             │
 │ TCGA-E2-A1B5-01 ┆ Basal-like            │
 │ TCGA-E2-A1B6-01 ┆ Luminal A             │
 │ TCGA-E2-A1BC-01 ┆ Luminal A             │
 │ TCGA-E2-A1BD-01 ┆ Luminal A             │
 └─────────────────┴───────────────────────┘,
 array([2, 2, 2, 2, 0, 3, 0, 2, 0, 0, 3, 0, 0, 2, 1, 1, 2, 1, 0, 2, 2, 2,
        3, 2, 2, 3, 0, 1, 0, 2, 3, 2, 2, 2, 1, 3, 2, 2, 2, 2, 2, 2, 3, 0,
        2, 3, 3, 0, 2, 0, 1, 0, 3, 3, 2, 2, 2, 2, 2, 2, 3, 3, 2, 2, 0

In [5]:
# ensure that the omic channels are alined with the labels and with each other
assert mrna.columns[1:] == cna.columns[1:] == mirna.columns[1:] == labels["sampleID"].to_list()

In [26]:
# TODO, join genes with identical features into a single vertex
def find_identical_rows(matrix):
    # Convert the matrix to a structured array
    structured_array = np.core.records.fromarrays(matrix.transpose())
    
    # Find unique rows and their corresponding labels
    _, labels = np.unique(structured_array, return_inverse=True)
    
    # Group indices by labels
    unique_labels, indices = np.unique(labels, return_index=True)
    identical_rows_indices = [np.where(labels == label)[0] for label in unique_labels]
    
    return identical_rows_indices

# Example usage
matrix = np.array([[1, 2], [3, 4], [1, 2], [5, 6], [3, 4]])
identical_rows_indices = find_identical_rows(matrix)

for indices in identical_rows_indices:
    print(f"Identical rows at indices: {indices}")

# identical_rows = find_identical_rows(cna_m)

Identical rows at indices: [0 2]
Identical rows at indices: [1 4]
Identical rows at indices: [3]


In [6]:
X_mrna = mrna[:,1:].to_numpy().T
X_cna = cna[:,1:].to_numpy().T
X_mirna = mirna[:,1:].to_numpy().T

X = np.hstack([X_mrna, X_cna, X_mirna])
X.shape

(483, 43982)

## Benchmarks

In [28]:
knn_eval(X, y, n_features=1000, test_size=0.4)

| KNN | 0.65 +/- 0.03 | 0.54 +/- 0.04 | 0.60 +/- 0.03 |
study.best_value=0.6013213222740035, study.best_params={'n_neighbors': 1}


In [29]:
svm_eval(X, y, n_trials=20, n_features_preselect=5000, n_features=1000, test_size=0.4, mode="linear")

Trial 1 / 20
Trial 2 / 20
Trial 3 / 20
Trial 4 / 20
Trial 5 / 20
Trial 6 / 20
Trial 7 / 20
Trial 8 / 20
Trial 9 / 20
Trial 10 / 20
Trial 11 / 20
Trial 12 / 20
Trial 13 / 20
Trial 14 / 20
Trial 15 / 20
Trial 16 / 20
Trial 17 / 20
Trial 18 / 20
Trial 19 / 20
Trial 20 / 20
| LIN SVM | 0.81 +/- 0.03 | 0.81 +/- 0.02 | 0.81 +/- 0.03 |
study.best_value=0.8142452090046813, study.best_params={'C': 0.001114063483643202, 'class_weight': None}


{'acc': 0.811340206185567,
 'f1_macro': 0.807820061683382,
 'f1_weighted': 0.8142452090046813,
 'acc_std': 0.029896907216494847,
 'f1_macro_std': 0.02454023617603619,
 'f1_weighted_std': 0.029062426848584122}

In [26]:
svm_eval(X, y, n_trials=20, n_features_preselect=5000, n_features=500, test_size=0.4, mode="rbf")

Trial 1 / 20


ValueError: when `importance_getter=='auto'`, the underlying estimator SVC should have `coef_` or `feature_importances_` attribute. Either pass a fitted estimator to feature selector or call fit before calling transform.

In [31]:
xgboost_eval(X, y, n_features=None, n_evals=3, test_size=0.4)

0 / 100
| XGBoost | 0.81 +/- 0.01 | 0.80 +/- 0.00 | 0.81 +/- 0.01 |
1 / 100
| XGBoost | 0.86 +/- 0.01 | 0.85 +/- 0.02 | 0.86 +/- 0.01 |
2 / 100
Pruning trial
3 / 100
| XGBoost | 0.86 +/- 0.02 | 0.86 +/- 0.01 | 0.86 +/- 0.02 |
4 / 100
Pruning trial
5 / 100
Pruning trial
6 / 100
Pruning trial
7 / 100
Pruning trial
8 / 100
Pruning trial
9 / 100
Pruning trial
10 / 100
Pruning trial
11 / 100
Pruning trial
12 / 100
Pruning trial
13 / 100
Pruning trial
14 / 100
Pruning trial
15 / 100
Pruning trial
16 / 100
Pruning trial
17 / 100
Pruning trial
18 / 100
Pruning trial
19 / 100
Pruning trial
20 / 100
Pruning trial
21 / 100
Pruning trial
22 / 100
23 / 100
Pruning trial
24 / 100
Pruning trial
25 / 100
Pruning trial
26 / 100
Pruning trial
27 / 100
Pruning trial
28 / 100
Pruning trial
29 / 100
Pruning trial
30 / 100
Pruning trial
31 / 100
Pruning trial
32 / 100
| XGBoost | 0.87 +/- 0.02 | 0.86 +/- 0.02 | 0.87 +/- 0.02 |
33 / 100
Pruning trial
34 / 100
35 / 100
36 / 100
Pruning trial
37 / 100
Pruning 

In [24]:
mlp_eval(X, y, n_trials=1, n_evals=5, n_features=20000, val_test_size=0.4)

Trial 0 / 1
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
study.best_value=2.342774343422382, study.best_params={}
| MLP | 0.79 +/- 0.04 | 0.77 +/- 0.07 | 0.78 +/- 0.04 |


In [14]:
np.array([0.85172786, 0.81978306, 0.83887178, 0.80676451, 0.78389554]).std()

0.023838376645362418

In [20]:
mlp_eval(
    X, y, n_trials=5, n_features=20000
)

Trial 0 / 5
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 1 / 5
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.7512744116330763 < 0.8205383457708766
Trial 2 / 5
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Trial 3 / 5
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.4000380364116356 < 0.8283377073551508
Trial 4 / 5
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 2 evals, cause 0.775808106627672 < 0.8283377073551508
study.best_value=2.4730962639824794, study.best_params={'lr': 0.001066748488589971, 'l1_lambda': 0.000808960415788037, 'dropout': 0.49978639342734293}
| MLP | 0.83 +/- 0.03 | 0.81 +/- 0.04 | 0.83 +/- 0.00 |


In [ ]:
# prepare data for BiGNN